# Net5 and Net6 Audio Models (Scipy-based)

This notebook is self-contained for Net5 (LSTM) and Net6 (LSTM + GAN feature augmentation) using original audio from `Data/genres_original`.


In [ ]:
# Cell 1: Imports and constants (scipy-only audio pipeline)
import os, glob, warnings, copy, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split
from scipy.io import wavfile
from scipy.signal import spectrogram, resample

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
NUM_CLASSES = 10
AUDIO_DIR = "Data/genres_original"
AUDIO_CLASS_NAMES = ["blues", "classical", "country", "disco", "hiphop", "jazz", "metal", "pop", "reggae", "rock"]
AUDIO_CLASS_TO_IDX = {class_name: index for index, class_name in enumerate(AUDIO_CLASS_NAMES)}
TARGET_SAMPLE_RATE = 22050
DURATION = 30
TARGET_NUM_SAMPLES = TARGET_SAMPLE_RATE * DURATION
N_FREQ_BINS = 128
MAX_AUDIO_TIME_STEPS = 1300
AUDIO_BATCH_SIZE = 16
AUDIO_DEBUG_EPOCHS = 1
AUDIO_FULL_EPOCHS = 20
EARLY_STOPPING_PATIENCE = 5
MIN_DELTA = 1e-4
GAN_DEBUG_EPOCHS = 1
GAN_FULL_EPOCHS = 20
AUDIO_LEARNING_RATE = 1e-3
GAN_LEARNING_RATE = 1e-4
NOISE_DIM = 100

print(f"DEVICE: {DEVICE}")
print(f"AUDIO_DIR: {AUDIO_DIR}")
print(f"AUDIO_CLASS_TO_IDX: {AUDIO_CLASS_TO_IDX}")


In [ ]:
# Cell 2: Utility functions for independent notebook execution

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


set_seed(SEED)


In [ ]:
# Cell 3: AudioSpectrogramDataset (replaces any previous MFCC-based dataset)
class AudioSpectrogramDataset(Dataset):
    def __init__(self, audio_dir: str):
        self.audio_dir = audio_dir
        self.class_names = AUDIO_CLASS_NAMES
        self.class_to_idx = AUDIO_CLASS_TO_IDX
        self.samples = []

        for class_name in self.class_names:
            class_dir = os.path.join(self.audio_dir, class_name)
            wav_paths = sorted(glob.glob(os.path.join(class_dir, "*.wav")))
            for wav_path in wav_paths:
                self.samples.append((wav_path, self.class_to_idx[class_name]))

    def __len__(self):
        return len(self.samples)

    @staticmethod
    def _to_float32_mono(audio_np: np.ndarray) -> np.ndarray:
        if audio_np.ndim > 1:
            audio_np = audio_np.mean(axis=1)

        if np.issubdtype(audio_np.dtype, np.integer):
            info = np.iinfo(audio_np.dtype)
            max_abs = max(abs(info.min), abs(info.max))
            audio_np = audio_np.astype(np.float32) / float(max_abs)
        else:
            audio_np = audio_np.astype(np.float32)

        return np.clip(audio_np, -1.0, 1.0)

    @staticmethod
    def _fit_length(waveform: np.ndarray, target_len: int) -> np.ndarray:
        if len(waveform) < target_len:
            pad_len = target_len - len(waveform)
            waveform = np.pad(waveform, (0, pad_len), mode="constant")
        elif len(waveform) > target_len:
            waveform = waveform[:target_len]
        return waveform

    @staticmethod
    def _fit_freq_bins(log_spec: np.ndarray, target_bins: int) -> np.ndarray:
        # log_spec shape: [freq_bins, time_steps]
        current_bins = log_spec.shape[0]
        if current_bins < target_bins:
            pad_bins = target_bins - current_bins
            log_spec = np.pad(log_spec, ((0, pad_bins), (0, 0)), mode="constant")
        elif current_bins > target_bins:
            log_spec = log_spec[:target_bins, :]
        return log_spec

    @staticmethod
    def _fit_time_steps(features_txf: np.ndarray, target_steps: int) -> np.ndarray:
        # features_txf shape: [time_steps, freq_bins]
        current_steps = features_txf.shape[0]
        if current_steps < target_steps:
            pad_steps = target_steps - current_steps
            features_txf = np.pad(features_txf, ((0, pad_steps), (0, 0)), mode="constant")
        elif current_steps > target_steps:
            features_txf = features_txf[:target_steps, :]
        return features_txf

    def __getitem__(self, idx: int):
        wav_path, label = self.samples[idx]

        try:
            sr, audio_np = wavfile.read(wav_path)
            audio_np = self._to_float32_mono(audio_np)

            if sr != TARGET_SAMPLE_RATE:
                target_len = int(round(len(audio_np) * TARGET_SAMPLE_RATE / sr))
                audio_np = resample(audio_np, target_len).astype(np.float32)
                sr = TARGET_SAMPLE_RATE

            audio_np = self._fit_length(audio_np, TARGET_NUM_SAMPLES)

            _, _, Sxx = spectrogram(
                audio_np,
                fs=sr,
                nperseg=1024,
                noverlap=512,
                mode="magnitude",
            )

            log_spec = np.log1p(Sxx).astype(np.float32)
            log_spec = self._fit_freq_bins(log_spec, N_FREQ_BINS)

            # [freq, time] -> [time, freq]
            features = log_spec.T
            features = self._fit_time_steps(features, MAX_AUDIO_TIME_STEPS)

            x = torch.tensor(features, dtype=torch.float32)
            y = torch.tensor(label, dtype=torch.long)
            return x, y

        except Exception as e:
            warnings.warn(f"Failed to load audio file {wav_path}: {e}")
            x = torch.zeros((MAX_AUDIO_TIME_STEPS, N_FREQ_BINS), dtype=torch.float32)
            y = torch.tensor(label, dtype=torch.long)
            return x, y


In [ ]:
# Cell 4: Audio dataset loading and 70/20/10 split with random_split
set_seed(SEED)
audio_dataset = AudioSpectrogramDataset(AUDIO_DIR)

print("Class names:", audio_dataset.class_names)
print("Class to index:", audio_dataset.class_to_idx)
print("Total audio samples:", len(audio_dataset))

n_total = len(audio_dataset)
n_train = int(0.7 * n_total)
n_val = int(0.2 * n_total)
n_test = n_total - n_train - n_val

split_generator = torch.Generator().manual_seed(SEED)
train_audio_dataset, val_audio_dataset, test_audio_dataset = random_split(
    audio_dataset,
    [n_train, n_val, n_test],
    generator=split_generator,
)

train_audio_loader = DataLoader(train_audio_dataset, batch_size=AUDIO_BATCH_SIZE, shuffle=True)
val_audio_loader = DataLoader(val_audio_dataset, batch_size=AUDIO_BATCH_SIZE, shuffle=False)
test_audio_loader = DataLoader(test_audio_dataset, batch_size=AUDIO_BATCH_SIZE, shuffle=False)

print(f"Train size: {len(train_audio_dataset)}")
print(f"Validation size: {len(val_audio_dataset)}")
print(f"Test size: {len(test_audio_dataset)}")

sample_x, sample_y = next(iter(train_audio_loader))
print("One feature batch shape:", tuple(sample_x.shape))
print("One label batch shape:", tuple(sample_y.shape))


In [ ]:
# Cell 5: Net5 LSTM model
class Net5LSTM(nn.Module):
    def __init__(self, input_size=N_FREQ_BINS, hidden_size=128, num_layers=2, num_classes=NUM_CLASSES):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.3,
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        # x: [batch, seq_len, N_FREQ_BINS]
        _, (h_n, _) = self.lstm(x)
        last_hidden = h_n[-1]  # [batch, hidden_size]
        logits = self.classifier(last_hidden)
        return logits


In [ ]:
# Cell 6: Audio training/evaluation with early stopping
criterion_audio = nn.CrossEntropyLoss()
audio_results_records = []


def train_one_epoch_audio(model, loader, optimizer, criterion, device=DEVICE):
    model.train()
    running_loss = 0.0
    total = 0

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        batch_size = xb.size(0)
        running_loss += loss.item() * batch_size
        total += batch_size

    return running_loss / max(total, 1)


def evaluate_audio(model, loader, criterion, device=DEVICE):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)

            batch_size = xb.size(0)
            running_loss += loss.item() * batch_size
            total += batch_size
            preds = torch.argmax(logits, dim=1)
            correct += (preds == yb).sum().item()

    avg_loss = running_loss / max(total, 1)
    accuracy_pct = (100.0 * correct / max(total, 1))
    return avg_loss, accuracy_pct


def train_audio_model(model, train_loader, val_loader, epochs, lr=AUDIO_LEARNING_RATE, device=DEVICE):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    history = {
        "train_loss": [],
        "val_loss": [],
        "val_accuracy": [],
    }

    best_state = copy.deepcopy(model.state_dict())
    best_val_loss = float("inf")
    best_val_acc = 0.0
    best_epoch = 0

    wait = 0
    early_stopped = False
    stopped_epoch = epochs

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch_audio(model, train_loader, optimizer, criterion_audio, device)
        val_loss, val_acc = evaluate_audio(model, val_loader, criterion_audio, device)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_acc)

        print(
            f"Epoch {epoch}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%"
        )

        improved = (best_val_loss - val_loss) >= MIN_DELTA
        if improved:
            best_val_loss = val_loss
            best_val_acc = val_acc
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= EARLY_STOPPING_PATIENCE:
                early_stopped = True
                stopped_epoch = epoch
                print(f"Early stopping triggered at epoch {epoch}.")
                break

    model.load_state_dict(best_state)

    final_val_loss, final_val_acc = evaluate_audio(model, val_loader, criterion_audio, device)

    summary = {
        "train_loss": history["train_loss"][-1] if history["train_loss"] else None,
        "val_loss": history["val_loss"][-1] if history["val_loss"] else None,
        "final_validation_accuracy": final_val_acc,
        "best_validation_accuracy": best_val_acc,
        "best_epoch": best_epoch,
        "stopped_epoch": stopped_epoch,
        "early_stopped": early_stopped,
        "history": history,
    }

    return model, summary


def test_audio_model(model, test_loader, criterion=criterion_audio, device=DEVICE):
    test_loss, test_acc = evaluate_audio(model, test_loader, criterion, device)
    return test_loss, test_acc


def append_audio_result(model_name, epochs, run_type, train_summary, test_loss, test_accuracy):
    audio_results_records.append(
        {
            "model": model_name,
            "epochs": epochs,
            "run_type": run_type,
            "train_loss": train_summary["train_loss"],
            "val_loss": train_summary["val_loss"],
            "final_validation_accuracy": train_summary["final_validation_accuracy"],
            "best_validation_accuracy": train_summary["best_validation_accuracy"],
            "best_epoch": train_summary["best_epoch"],
            "stopped_epoch": train_summary["stopped_epoch"],
            "early_stopped": train_summary["early_stopped"],
            "test_loss": test_loss,
            "test_accuracy": test_accuracy,
        }
    )


In [ ]:
# Cell 7: Net5 smoke test + debug run (enabled by default)
def smoke_test_audio_model():
    model = Net5LSTM().to(DEVICE)
    xb, _ = next(iter(train_audio_loader))
    xb = xb.to(DEVICE)
    with torch.no_grad():
        out = model(xb)
    assert out.shape == (xb.size(0), NUM_CLASSES), f"Unexpected output shape: {out.shape}"
    print("Smoke test output shape:", tuple(out.shape))
    print("Net5 trainable parameters:", count_parameters(model))


smoke_test_audio_model()

RUN_DEBUG_NET5 = True
RUN_FULL_NET5 = False

net5_debug_model = None
net5_debug_summary = None

if RUN_DEBUG_NET5:
    print("\nRunning Net5 debug training...")
    set_seed(SEED)
    net5_debug_model = Net5LSTM()
    net5_debug_model, net5_debug_summary = train_audio_model(
        net5_debug_model,
        train_audio_loader,
        val_audio_loader,
        epochs=AUDIO_DEBUG_EPOCHS,
    )
    net5_debug_test_loss, net5_debug_test_acc = test_audio_model(net5_debug_model, test_audio_loader)
    append_audio_result(
        model_name="Net5LSTM",
        epochs=AUDIO_DEBUG_EPOCHS,
        run_type="debug",
        train_summary=net5_debug_summary,
        test_loss=net5_debug_test_loss,
        test_accuracy=net5_debug_test_acc,
    )
    print(f"Net5 debug test loss: {net5_debug_test_loss:.4f}")
    print(f"Net5 debug test accuracy: {net5_debug_test_acc:.2f}%")


In [ ]:
# Cell 8: Net5 full training (disabled by default)
RUN_FULL_NET5 = False

net5_full_model = None
net5_full_summary = None

if RUN_FULL_NET5:
    print("\nRunning Net5 full training...")
    set_seed(SEED)
    net5_full_model = Net5LSTM()
    net5_full_model, net5_full_summary = train_audio_model(
        net5_full_model,
        train_audio_loader,
        val_audio_loader,
        epochs=AUDIO_FULL_EPOCHS,
    )
    net5_full_test_loss, net5_full_test_acc = test_audio_model(net5_full_model, test_audio_loader)
    append_audio_result(
        model_name="Net5LSTM",
        epochs=AUDIO_FULL_EPOCHS,
        run_type="full",
        train_summary=net5_full_summary,
        test_loss=net5_full_test_loss,
        test_accuracy=net5_full_test_acc,
    )
    print(f"Net5 full test loss: {net5_full_test_loss:.4f}")
    print(f"Net5 full test accuracy: {net5_full_test_acc:.2f}%")


In [ ]:
# Cell 9: GAN dataset preparation from real training spectrogram features
real_feature_batches = []
real_label_batches = []

for xb, yb in train_audio_loader:
    real_feature_batches.append(xb)
    real_label_batches.append(yb)

real_train_features = torch.cat(real_feature_batches, dim=0)  # [N, T, F]
train_labels_tensor = torch.cat(real_label_batches, dim=0)    # [N]

GAN_FEATURE_DIM = MAX_AUDIO_TIME_STEPS * N_FREQ_BINS
real_train_features_flat = real_train_features.view(real_train_features.size(0), -1)

gan_train_dataset = TensorDataset(real_train_features_flat, train_labels_tensor)
gan_train_loader = DataLoader(gan_train_dataset, batch_size=AUDIO_BATCH_SIZE, shuffle=True)

print("Real training feature tensor shape:", tuple(real_train_features.shape))
print("Flattened training feature tensor shape:", tuple(real_train_features_flat.shape))
print("GAN_FEATURE_DIM:", GAN_FEATURE_DIM)


In [ ]:
# Cell 10: Feature-level GAN models (spectrogram feature vectors)
class SpectrogramGenerator(nn.Module):
    def __init__(self, noise_dim=NOISE_DIM, output_dim=GAN_FEATURE_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(noise_dim, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, output_dim),
        )

    def forward(self, z):
        return self.net(z)


class SpectrogramDiscriminator(nn.Module):
    def __init__(self, input_dim=GAN_FEATURE_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)


In [ ]:
# Cell 11: GAN training function + debug run (enabled by default)
def train_gan(generator, discriminator, gan_loader, epochs):
    generator = generator.to(DEVICE)
    discriminator = discriminator.to(DEVICE)

    criterion_gan = nn.BCELoss()
    optimizer_g = optim.Adam(generator.parameters(), lr=GAN_LEARNING_RATE)
    optimizer_d = optim.Adam(discriminator.parameters(), lr=GAN_LEARNING_RATE)

    for epoch in range(1, epochs + 1):
        d_epoch_loss = 0.0
        g_epoch_loss = 0.0
        n_batches = 0

        for batch in gan_loader:
            real_x = batch[0].to(DEVICE)
            batch_size = real_x.size(0)

            valid = torch.ones((batch_size, 1), device=DEVICE)
            fake = torch.zeros((batch_size, 1), device=DEVICE)

            # Discriminator step
            optimizer_d.zero_grad()
            real_pred = discriminator(real_x)
            d_real_loss = criterion_gan(real_pred, valid)

            z = torch.randn(batch_size, NOISE_DIM, device=DEVICE)
            gen_x = generator(z)
            fake_pred = discriminator(gen_x.detach())
            d_fake_loss = criterion_gan(fake_pred, fake)

            d_loss = 0.5 * (d_real_loss + d_fake_loss)
            d_loss.backward()
            optimizer_d.step()

            # Generator step
            optimizer_g.zero_grad()
            z = torch.randn(batch_size, NOISE_DIM, device=DEVICE)
            gen_x = generator(z)
            fooled_pred = discriminator(gen_x)
            g_loss = criterion_gan(fooled_pred, valid)
            g_loss.backward()
            optimizer_g.step()

            d_epoch_loss += d_loss.item()
            g_epoch_loss += g_loss.item()
            n_batches += 1

        print(
            f"GAN Epoch {epoch}/{epochs} | "
            f"D loss: {d_epoch_loss / max(n_batches, 1):.4f} | "
            f"G loss: {g_epoch_loss / max(n_batches, 1):.4f}"
        )

    return generator, discriminator


RUN_DEBUG_GAN = True
RUN_FULL_GAN = False

generator = SpectrogramGenerator()
discriminator = SpectrogramDiscriminator()

if RUN_DEBUG_GAN:
    print("\nRunning GAN debug training...")
    set_seed(SEED)
    generator, discriminator = train_gan(generator, discriminator, gan_train_loader, GAN_DEBUG_EPOCHS)

    with torch.no_grad():
        z = torch.randn(8, NOISE_DIM, device=DEVICE)
        synth_flat = generator(z)
        synth_seq = synth_flat.view(-1, MAX_AUDIO_TIME_STEPS, N_FREQ_BINS)
    print("Generated debug synthetic tensor shape:", tuple(synth_seq.shape))


In [ ]:
# Cell 12: Build augmented training dataset for Net6

def generate_synthetic_spectrograms(generator_model, num_samples):
    generator_model.eval()
    synth_parts = []
    remaining = num_samples
    with torch.no_grad():
        while remaining > 0:
            bsz = min(AUDIO_BATCH_SIZE, remaining)
            z = torch.randn(bsz, NOISE_DIM, device=DEVICE)
            synth_flat = generator_model(z)
            synth_seq = synth_flat.view(-1, MAX_AUDIO_TIME_STEPS, N_FREQ_BINS).cpu()
            synth_parts.append(synth_seq)
            remaining -= bsz

    if len(synth_parts) == 0:
        return torch.empty((0, MAX_AUDIO_TIME_STEPS, N_FREQ_BINS), dtype=torch.float32)

    return torch.cat(synth_parts, dim=0).float()


if RUN_FULL_GAN:
    synthetic_count = len(train_audio_dataset)
else:
    synthetic_count = min(100, len(train_audio_dataset))

synthetic_features = generate_synthetic_spectrograms(generator, synthetic_count)
if synthetic_count > 0:
    sample_indices = torch.randint(0, len(train_labels_tensor), (synthetic_count,))
    synthetic_labels = train_labels_tensor[sample_indices].clone()
else:
    synthetic_labels = torch.empty((0,), dtype=torch.long)

augmented_features = torch.cat([real_train_features.float(), synthetic_features], dim=0)
augmented_labels = torch.cat([train_labels_tensor.long(), synthetic_labels.long()], dim=0)

augmented_train_dataset = TensorDataset(augmented_features, augmented_labels)
augmented_train_audio_loader = DataLoader(augmented_train_dataset, batch_size=AUDIO_BATCH_SIZE, shuffle=True)

print(f"Number of real training samples: {len(real_train_features)}")
print(f"Number of synthetic samples: {len(synthetic_features)}")
print(f"Number of augmented samples: {len(augmented_train_dataset)}")


In [ ]:
# Cell 13: Net6 training (same architecture as Net5, trained on augmented data)
RUN_DEBUG_NET6 = True
RUN_FULL_NET6 = False

net6_debug_model = None
net6_full_model = None

if RUN_DEBUG_NET6:
    print("\nRunning Net6 debug training (augmented data)...")
    set_seed(SEED)
    net6_debug_model = Net5LSTM()
    net6_debug_model, net6_debug_summary = train_audio_model(
        net6_debug_model,
        augmented_train_audio_loader,
        val_audio_loader,
        epochs=AUDIO_DEBUG_EPOCHS,
    )
    net6_debug_test_loss, net6_debug_test_acc = test_audio_model(net6_debug_model, test_audio_loader)
    append_audio_result(
        model_name="Net6LSTM_GANAug",
        epochs=AUDIO_DEBUG_EPOCHS,
        run_type="debug",
        train_summary=net6_debug_summary,
        test_loss=net6_debug_test_loss,
        test_accuracy=net6_debug_test_acc,
    )
    print(f"Net6 debug test loss: {net6_debug_test_loss:.4f}")
    print(f"Net6 debug test accuracy: {net6_debug_test_acc:.2f}%")

if RUN_FULL_NET6:
    print("\nRunning Net6 full training (augmented data)...")
    set_seed(SEED)
    net6_full_model = Net5LSTM()
    net6_full_model, net6_full_summary = train_audio_model(
        net6_full_model,
        augmented_train_audio_loader,
        val_audio_loader,
        epochs=AUDIO_FULL_EPOCHS,
    )
    net6_full_test_loss, net6_full_test_acc = test_audio_model(net6_full_model, test_audio_loader)
    append_audio_result(
        model_name="Net6LSTM_GANAug",
        epochs=AUDIO_FULL_EPOCHS,
        run_type="full",
        train_summary=net6_full_summary,
        test_loss=net6_full_test_loss,
        test_accuracy=net6_full_test_acc,
    )
    print(f"Net6 full test loss: {net6_full_test_loss:.4f}")
    print(f"Net6 full test accuracy: {net6_full_test_acc:.2f}%")


In [ ]:
# Cell 14: Results table and CSV export
results_audio_df = pd.DataFrame(audio_results_records)
results_audio_df

results_audio_df.to_csv("results_audio_models.csv", index=False)
print("Saved results to results_audio_models.csv")


In [ ]:
# Cell 15: Optional overnight run (disabled by default)
RUN_OVERNIGHT_NET5_NET6 = False

if RUN_OVERNIGHT_NET5_NET6:
    print("Starting overnight Net5 + GAN + Net6 full pipeline...")
    set_seed(SEED)

    # 1) Net5 full
    overnight_net5 = Net5LSTM()
    overnight_net5, overnight_net5_summary = train_audio_model(
        overnight_net5,
        train_audio_loader,
        val_audio_loader,
        epochs=AUDIO_FULL_EPOCHS,
    )
    overnight_net5_test_loss, overnight_net5_test_acc = test_audio_model(overnight_net5, test_audio_loader)
    append_audio_result(
        model_name="Net5LSTM",
        epochs=AUDIO_FULL_EPOCHS,
        run_type="overnight_full",
        train_summary=overnight_net5_summary,
        test_loss=overnight_net5_test_loss,
        test_accuracy=overnight_net5_test_acc,
    )

    # 2) GAN full
    overnight_generator = SpectrogramGenerator()
    overnight_discriminator = SpectrogramDiscriminator()
    overnight_generator, overnight_discriminator = train_gan(
        overnight_generator,
        overnight_discriminator,
        gan_train_loader,
        GAN_FULL_EPOCHS,
    )

    # 3) Generate synthetic spectrograms
    overnight_synth_features = generate_synthetic_spectrograms(overnight_generator, len(train_audio_dataset))
    overnight_idx = torch.randint(0, len(train_labels_tensor), (len(overnight_synth_features),))
    overnight_synth_labels = train_labels_tensor[overnight_idx]

    overnight_aug_features = torch.cat([real_train_features.float(), overnight_synth_features], dim=0)
    overnight_aug_labels = torch.cat([train_labels_tensor.long(), overnight_synth_labels.long()], dim=0)
    overnight_aug_loader = DataLoader(
        TensorDataset(overnight_aug_features, overnight_aug_labels),
        batch_size=AUDIO_BATCH_SIZE,
        shuffle=True,
    )

    # 4) Net6 full
    overnight_net6 = Net5LSTM()
    overnight_net6, overnight_net6_summary = train_audio_model(
        overnight_net6,
        overnight_aug_loader,
        val_audio_loader,
        epochs=AUDIO_FULL_EPOCHS,
    )
    overnight_net6_test_loss, overnight_net6_test_acc = test_audio_model(overnight_net6, test_audio_loader)
    append_audio_result(
        model_name="Net6LSTM_GANAug",
        epochs=AUDIO_FULL_EPOCHS,
        run_type="overnight_full",
        train_summary=overnight_net6_summary,
        test_loss=overnight_net6_test_loss,
        test_accuracy=overnight_net6_test_acc,
    )

    # 5) Save results
    overnight_df = pd.DataFrame(audio_results_records)
    overnight_df.to_csv("results_audio_models.csv", index=False)

    # 6) Print final summary table
    print("Overnight run complete. Final summary:")
    print(overnight_df)
